In [6]:

# Read Bronze and convert data types
typed = spark.sql("""
SELECT
    TRIM(Invoice) AS Invoice,
    TRIM(StockCode) AS StockCode,
    Description,
    TRY_CAST(Quantity AS INT) AS Quantity,
    TRY_CAST(InvoiceDate AS TIMESTAMP) AS InvoiceDate,
    TRY_CAST(Price AS DECIMAL(18,6)) AS Price,
    NULLIF(TRIM(CustomerID), '') AS CustomerID,
    TRIM(Country) AS Country,
    SourceSheet,
    SourceFile
FROM bronze.retail_transactions
""")

typed.createOrReplaceTempView("retail_typed")

# Classify transactions without deleting rows
silver = spark.sql("""
SELECT *,
    CASE
        WHEN Invoice IS NULL
          OR Quantity IS NULL
          OR Price IS NULL
          OR InvoiceDate IS NULL
            THEN 'InvalidFields'
        WHEN StockCode = 'B'
          AND Price < 0
          AND LOWER(TRIM(Description)) = 'adjust bad debt'
            THEN 'BadDebtAdjustment'
        WHEN UPPER(Invoice) LIKE 'C%' AND Quantity < 0
            THEN 'Cancelled'
        WHEN UPPER(Invoice) LIKE 'C%'
            THEN 'CancellationAnomaly'
        WHEN Quantity < 0
            THEN 'NegativeWithoutC'
        WHEN Quantity = 0
            THEN 'ZeroQuantity'
        ELSE 'RegularSale'
    END AS TransactionClass,

    CustomerID IS NULL AS MissingCustomerID,
    Price = 0 AS ZeroPrice,
    Quantity * Price AS LineAmount

FROM retail_typed
""")

# Store as a Delta table
silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.retail_transactions")

print("Silver table created.")


StatementMeta(, d6ecc2cc-a2f5-4466-b93c-e24b15e0fcf5, 8, Finished, Available, Finished, False)

Silver table created.


In [2]:

display(spark.sql("""
SELECT
    TransactionClass,
    COUNT(*) AS RowCount
FROM silver.retail_transactions
GROUP BY TransactionClass
ORDER BY RowCount DESC
"""))


StatementMeta(, d6ecc2cc-a2f5-4466-b93c-e24b15e0fcf5, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e452f84-6003-4fc7-bd9b-c104a7704aeb)

In [3]:

display(spark.sql("""
SELECT
    TransactionClass,
    COUNT(*) AS RowCount,
    ROUND(SUM(LineAmount), 2) AS NetAmount,
    ROUND(SUM(
        CASE WHEN LineAmount > 0
        THEN LineAmount ELSE 0 END
    ), 2) AS PositiveAmount
FROM silver.retail_transactions
GROUP BY TransactionClass
ORDER BY RowCount DESC
"""))


StatementMeta(, d6ecc2cc-a2f5-4466-b93c-e24b15e0fcf5, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 19060e6c-2105-4138-b126-48b92110d6c5)

In [4]:

display(spark.sql("""
SELECT
    TransactionClass,
    COUNT(*) AS RowCount,

    SUM(CASE WHEN Price < 0
        THEN 1 ELSE 0 END) AS NegativePriceRows,

    SUM(CASE WHEN Price = 0
        THEN 1 ELSE 0 END) AS ZeroPriceRows,

    SUM(CASE WHEN LineAmount < 0
        THEN 1 ELSE 0 END) AS NegativeAmountRows,

    ROUND(SUM(CASE WHEN LineAmount < 0
        THEN LineAmount ELSE 0 END), 2) AS NegativeAmount

FROM silver.retail_transactions
GROUP BY TransactionClass
ORDER BY RowCount DESC
"""))


StatementMeta(, d6ecc2cc-a2f5-4466-b93c-e24b15e0fcf5, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2529eb6d-b636-436c-834d-b615a58740a4)

In [7]:

display(spark.sql("""
SELECT
    Invoice,
    StockCode,
    Description,
    Quantity,
    Price,
    LineAmount,
    TransactionClass
FROM silver.retail_transactions
WHERE Price < 0
   OR TransactionClass = 'CancellationAnomaly'
ORDER BY Price, Invoice
"""))


StatementMeta(, d6ecc2cc-a2f5-4466-b93c-e24b15e0fcf5, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c36a9ba-00de-40ad-87f1-f9e1b987ac0a)